# M15 v4 — Cross-Task Consistency Scorer on Real ICBHI Data (CORE NOVELTY)

**Model ID:** M15 (The Core Novelty)  
**Model Name:** Cross-Task Consistency Scorer (OWL Stage 0 → 1)  
**Member:** B — Disease Diagnosis & Staged Open-World Learning Lead  
**Project:** OWMTL — Cluster-Aware Open-World Multi-Task Learning for Respiratory Sound and Disease Diagnosis  
**Requires:** M2 Backbone Checkpoint (`best_model.pth`), M13 Prototypical Disease Head Checkpoint (`best_model.pth`)  

---

### Why M15 Uses M13 Checkpoint
M15 is the **Cross-Task Consistency Scorer**. It measures the structural disagreement between:
1. **M2 Sound Event Head:** Acoustic cycle predictions (Normal, Crackle, Wheeze, Both) mapped to an *implied* disease profile.
2. **M13 Prototypical Disease Head:** Clinical patient predictions over known classes (COPD, Healthy, URTI).
High disagreement $S_{disagree}(x) = \| P_{sound\_implied}(x) - P_{disease}(x) \|_2$ indicates an **Unknown Disease** (Stage 1 OWL).

### v4 Core Improvements
1. **REAL ICBHI Audio Loading:** Extracts log-mel spectrograms from 104 Known Patients (COPD, Healthy, URTI) and 19 Unknown Patients (Pneumonia, Bronchiectasis, Bronchiolitis).
2. **Patient-Level Evaluation:** Aggregates cycle-level disagreement scores per patient for true patient-independent diagnostic evaluation.
3. **M29 Baseline Comparison:** Benchmarks AUROC, AUPR, and FPR95 directly against M29 Energy OOD score (0.6466).
4. **Protocol §4 Compliance:** Outputs full metrics suite in `results_M15.json` with Base64 HTML download button.


## Section 1: Setup & Dependencies


In [1]:
# ============================================================
# Section 1: Environment Setup & Dependencies
# ============================================================
import os
import sys
import re
import json
import math
import time
import glob
import copy
import random
import warnings
import datetime
import tempfile
import base64
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')  # Non-interactive backend for Colab/Kaggle
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from sklearn.metrics import (
    roc_curve, auc, precision_recall_curve, average_precision_score,
    confusion_matrix, accuracy_score
)

warnings.filterwarnings('ignore')

# ---- Reproducibility ----
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
GPU_NAME = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'

print(f'Device:  {DEVICE} ({GPU_NAME})')
print(f'PyTorch: {torch.__version__}')
print(f'Python:  {sys.version.split()[0]}')


Device:  cuda (Tesla T4)
PyTorch: 2.11.0+cu128
Python:  3.12.13


## Section 2: Configuration & Path Resolution


In [2]:
# ============================================================
# Section 2: Configuration & Path Resolution
# ============================================================

# ---- Auto-detect Platform ----
if os.path.exists('/content'):
    PLATFORM = 'Colab'
    BASE_DIR = '/content'
elif os.path.exists('/kaggle'):
    PLATFORM = 'Kaggle'
    BASE_DIR = '/kaggle/working'
else:
    PLATFORM = 'Local'
    BASE_DIR = '.'

print(f'Platform: {PLATFORM}')

# ---- Google Drive Mount (Colab) ----
DRIVE_DIR = None
if PLATFORM == 'Colab':
    try:
        from google.colab import drive
        drive.mount('/content/drive', force_remount=False)
        DRIVE_DIR = '/content/drive/MyDrive/OWMTL/M15'
        os.makedirs(DRIVE_DIR, exist_ok=True)
        print(f'Drive Backup Path: {DRIVE_DIR}')
    except Exception as e:
        print(f'Drive mount skipped ({e})')

# ---- ICBHI Dataset Path Resolution ----
POSSIBLE_ROOTS = [
    "/kaggle/input/respiratory-sound-database/Respiratory_Sound_Database/Respiratory_Sound_Database/audio_and_txt_files",
    "/kaggle/input/respiratory-sound-database/audio_and_txt_files",
    "/kaggle/input/respiratory-sound-database/Respiratory_Sound_Database/audio_and_txt_files",
    "/kaggle/input/vbookshelf/respiratory-sound-database/Respiratory_Sound_Database/Respiratory_Sound_Database/audio_and_txt_files",
    "/kaggle/input/icbhi-2017-respiratory-sound-database/audio_and_txt_files",
    "/kaggle/input/icbhi2017/audio_and_txt_files",
    "/content/Respiratory_Sound_Database/Respiratory_Sound_Database/audio_and_txt_files",
    "/content/drive/MyDrive/respiratory-sound-database/audio_and_txt_files",
    "/content/drive/MyDrive/OWMTL/data/audio_and_txt_files",
    "./data/audio_and_txt_files",
]
DATA_ROOT = next((p for p in POSSIBLE_ROOTS if os.path.exists(p)), None)

if DATA_ROOT is None and os.path.exists("/kaggle/input"):
    for root, dirs, files in os.walk("/kaggle/input"):
        if any(f.endswith(".wav") for f in files) and any(f.endswith(".txt") for f in files):
            DATA_ROOT = root
            print(f"Dynamic Kaggle dataset resolution found: {DATA_ROOT}")
            break

# ---- Automatic Kaggle Dataset Downloader for Colab ----
if DATA_ROOT is None and PLATFORM == 'Colab':
    print('\n📥 ICBHI dataset not found in Colab. Checking Kaggle credentials...')
    drive_kjson = '/content/drive/MyDrive/kaggle.json'
    if os.path.exists(drive_kjson):
        print(f'Found kaggle.json in Google Drive: {drive_kjson}')
        os.system('mkdir -p ~/.kaggle && cp /content/drive/MyDrive/kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json')
    elif not os.path.exists(os.path.expanduser('~/.kaggle/kaggle.json')):
        try:
            from google.colab import files
            print('Please upload your kaggle.json file:')
            uploaded = files.upload()
            if 'kaggle.json' in uploaded:
                os.system('mkdir -p ~/.kaggle && mv kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json')
        except Exception as e:
            print(f'Upload prompt skipped: {e}')

    if os.path.exists(os.path.expanduser('~/.kaggle/kaggle.json')):
        print('Downloading ICBHI dataset from Kaggle into /content...')
        os.system('pip install -q kaggle')
        os.system('kaggle datasets download -d vbookshelf/respiratory-sound-database -p /content --unzip')
        DATA_ROOT = next((p for p in POSSIBLE_ROOTS if os.path.exists(p)), None)
        if DATA_ROOT is None and os.path.exists('/content'):
            for root, dirs, files in os.walk('/content'):
                if any(f.endswith('.wav') for f in files) and any(f.endswith('.txt') for f in files):
                    DATA_ROOT = root
                    break

if DATA_ROOT and os.path.exists(DATA_ROOT):
    print(f'✅ ICBHI dataset verified: {DATA_ROOT}')
else:
    print(f'⚠️ DATA_ROOT set to fallback path: {DATA_ROOT}')

# ---- Model Checkpoint Resolution ----
# Set custom file paths below if uploaded under custom names!
M2_MANUAL_PATH = None   # e.g., '/content/M2_best_model.pth'
M13_MANUAL_PATH = None  # e.g., '/content/M13_best_model.pth'

M2_CKPT_CANDIDATES = [
    M2_MANUAL_PATH,
    '/content/M2_best_model.pth',
    '/content/best_model.pth',
    '/content/drive/MyDrive/OWMTL/M2/best_model.pth',
    '/content/drive/MyDrive/OWMTL/M2/checkpoints/best_model.pth',
    '/content/drive/MyDrive/OWMTL/M2/results/best_model.pth',
    '/kaggle/input/m2-checkpoint/best_model.pth',
    '/kaggle/input/owmtl-m2/best_model.pth',
    '/kaggle/input/m2-best-model/best_model.pth',
    '/kaggle/input/m2-results/best_model.pth',
    '../M2/best_model.pth',
    '../M2/checkpoints/best_model.pth',
    '../M2/results/best_model.pth',
    os.path.join(BASE_DIR, 'checkpoints', 'best_model.pth'),
    os.path.join(BASE_DIR, 'best_model.pth'),
    './best_model.pth',
]
M2_CKPT_PATH = next((p for p in M2_CKPT_CANDIDATES if p and os.path.exists(p)), None)

if M2_CKPT_PATH is None and os.path.exists('/kaggle/input'):
    for root, dirs, files in os.walk('/kaggle/input'):
        for f in files:
            if 'm2' in f.lower() or 'm2' in root.lower():
                if f.endswith('.pth'):
                    M2_CKPT_PATH = os.path.join(root, f)
                    print(f'Dynamic Kaggle M2 checkpoint resolution found: {M2_CKPT_PATH}')
                    break
        if M2_CKPT_PATH: break

M13_CKPT_CANDIDATES = [
    M13_MANUAL_PATH,
    '/content/M13_best_model.pth',
    '/content/best_model.pth',
    '/content/drive/MyDrive/OWMTL/M13/best_model.pth',
    '/content/drive/MyDrive/OWMTL/M13/results_M13/best_model.pth',
    '/kaggle/input/m13-checkpoint/best_model.pth',
    '/kaggle/input/owmtl-m13/best_model.pth',
    '/kaggle/input/m13-best-model/best_model.pth',
    '/kaggle/input/m13-results/best_model.pth',
    '../M13/results_M13/best_model.pth',
    '../M13/best_model.pth',
    './results_M13/best_model.pth',
    os.path.join(BASE_DIR, 'results_M13', 'best_model.pth'),
    os.path.join(BASE_DIR, 'best_model.pth'),
]
M13_CKPT_PATH = next((p for p in M13_CKPT_CANDIDATES if p and os.path.exists(p)), None)

if M13_CKPT_PATH is None and os.path.exists('/kaggle/input'):
    for root, dirs, files in os.walk('/kaggle/input'):
        for f in files:
            if 'm13' in f.lower() or 'm13' in root.lower():
                if f.endswith('.pth'):
                    M13_CKPT_PATH = os.path.join(root, f)
                    print(f'Dynamic Kaggle M13 checkpoint resolution found: {M13_CKPT_PATH}')
                    break
        if M13_CKPT_PATH: break

CFG = {
    'model_id': 'M15',
    'model_name': 'Cross-Task Consistency Scorer — v4',
    'member': 'B',
    'member_name': 'Member B (Disease Diagnosis & OWL)',
    'seed': SEED,

    # Shared Audio Parameters
    'sample_rate': 16000,
    'duration_s': 8.0,
    'n_mels': 128,
    'n_fft': 1024,
    'hop_length': 160,
    'win_length': 400,
    'f_min': 50,
    'f_max': 2000,

    'n_samples': int(16000 * 8.0),
    'n_frames': 1 + math.floor(128000 / 160),

    # Known Classes (Stage 0) vs Unknown Classes (Stage 1)
    'disease_classes': ['COPD', 'Healthy', 'URTI'],
    'unknown_disease_classes': ['Pneumonia', 'Bronchiectasis', 'Bronchiolitis'],
    'sound_classes': ['Normal', 'Crackle', 'Wheeze', 'Both'],

    'batch_size': 32,
    'm2_depth': 5,
    'm2_base_width': 48,
    'm2_dropout': 0.4,
    'm2_fc_dim': 128,
    'proto_embed_dim': 256,
    'proto_temperature': 0.1,

    'data_root': DATA_ROOT,
    'm2_ckpt_path': M2_CKPT_PATH,
    'm13_ckpt_path': M13_CKPT_PATH,
    'results_dir': os.path.join(BASE_DIR, 'results_M15'),
}

os.makedirs(CFG['results_dir'], exist_ok=True)

print(f"\n{'='*60}")
print("M15 v4 CONFIGURATION — Cross-Task Consistency Scorer")
print(f"{'='*60}")
print(f"  M2 Checkpoint:  {CFG['m2_ckpt_path'] or 'NOT FOUND (Upload M2 best_model.pth)'}")
print(f"  M13 Checkpoint: {CFG['m13_ckpt_path'] or 'NOT FOUND (Upload M13 best_model.pth)'}")
print(f"  Data Root:      {CFG['data_root']}")
print(f"{'='*60}")


Platform: Colab
Mounted at /content/drive
Drive Backup Path: /content/drive/MyDrive/OWMTL/M15

📥 ICBHI dataset not found in Colab. Checking Kaggle credentials...
Please upload your kaggle.json file:


Saving kaggle.json to kaggle.json
✅ ICBHI dataset verified: /content/Respiratory_Sound_Database/Respiratory_Sound_Database/audio_and_txt_files

M15 v4 CONFIGURATION — Cross-Task Consistency Scorer
  M2 Checkpoint:  /content/M2_best_model.pth
  M13 Checkpoint: /content/M13_best_model.pth
  Data Root:      /content/Respiratory_Sound_Database/Respiratory_Sound_Database/audio_and_txt_files


## Section 3: ICBHI Audio Loading — Known vs Unknown Patients


In [3]:
# ============================================================
# Section 3: ICBHI Audio Loading — Known vs Unknown Patients
# ============================================================

ICBHI_KNOWN_DISEASES = {'COPD': 0, 'Healthy': 1, 'URTI': 2}
ICBHI_UNKNOWN_DISEASES = {'Pneumonia': -1, 'Bronchiectasis': -1, 'Bronchiolitis': -1, 'Asthma': -1, 'LRTI': -1}

try:
    import librosa
except ImportError:
    import subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "librosa"])
    import librosa

def extract_log_mel(wav_path, start, end, cfg):
    """Extract log-mel spectrogram matching M2/M13 preprocessing."""
    sr, n_samples = cfg['sample_rate'], cfg['n_samples']
    try:
        audio, _ = librosa.load(wav_path, sr=sr, offset=start, duration=max(end - start, 0.05), mono=True)
    except Exception:
        return np.zeros((1, cfg['n_mels'], cfg['n_frames']), dtype=np.float32)

    if len(audio) == 0:
        return np.zeros((1, cfg['n_mels'], cfg['n_frames']), dtype=np.float32)

    if len(audio) < n_samples:
        reps = math.ceil(n_samples / len(audio))
        audio = np.tile(audio, reps)[:n_samples]
    else:
        audio = audio[:n_samples]

    mel = librosa.feature.melspectrogram(
        y=audio, sr=sr, n_mels=cfg['n_mels'], n_fft=cfg['n_fft'],
        hop_length=cfg['hop_length'], win_length=cfg['win_length'],
        fmin=cfg['f_min'], fmax=cfg['f_max'], power=2.0)
    log_mel = librosa.power_to_db(mel, ref=np.max)
    log_mel = (log_mel - log_mel.min()) / (log_mel.max() - log_mel.min() + 1e-8)

    T = log_mel.shape[1]
    if T < cfg['n_frames']:
        log_mel = np.pad(log_mel, ((0, 0), (0, cfg['n_frames'] - T)), mode='constant')
    else:
        log_mel = log_mel[:, :cfg['n_frames']]

    return log_mel[np.newaxis, :, :].astype(np.float32)

def parse_annotation_file(txt_path):
    cycles = []
    with open(txt_path, 'r') as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) < 4: continue
            try:
                start, end = float(parts[0]), float(parts[1])
                crackle, wheeze = int(parts[2]), int(parts[3])
            except ValueError: continue
            if end <= start: continue
            if crackle == 0 and wheeze == 0: label = 0
            elif crackle == 1 and wheeze == 0: label = 1
            elif crackle == 0 and wheeze == 1: label = 2
            else: label = 3
            cycles.append({'start': start, 'end': end, 'label': label})
    return cycles

def load_diagnosis_map(data_root):
    target_names = ["patient_diagnosis.csv", "ICBHI_Challenge_diagnosis.txt", "patient_diagnosis.txt"]
    candidates = []
    curr = data_root
    for _ in range(4):
        for name in target_names: candidates.append(os.path.join(curr, name))
        parent = os.path.dirname(curr)
        if parent == curr: break
        curr = parent
    if os.path.exists("/kaggle/input"):
        for root, dirs, files in os.walk("/kaggle/input"):
            for name in target_names:
                if name in files: candidates.append(os.path.join(root, name))

    for path in candidates:
        if not os.path.exists(path): continue
        diag_map = {}
        with open(path, 'r', encoding='utf-8', errors='ignore') as f:
            for line in f:
                line_str = line.strip()
                if not line_str: continue
                parts = [p.strip() for p in re.split(r'[,;\t\s]+', line_str) if p.strip()]
                if len(parts) >= 2:
                    try:
                        pid = int(parts[0])
                        disease = parts[1]
                        diag_map[pid] = disease
                    except ValueError: continue
        if diag_map:
            print(f"Loaded diagnosis map: {path} ({len(diag_map)} patients)")
            return diag_map
    return None

def build_owl_dataset(data_root, cfg):
    wav_paths = sorted(glob.glob(os.path.join(data_root, "*.wav")))
    if not wav_paths:
        raise FileNotFoundError(f"No .wav files under {data_root}.")
    diag_map = load_diagnosis_map(data_root)
    if diag_map is None:
        raise FileNotFoundError(f"Diagnosis map not found.")

    known_rows, unknown_rows = [], []

    for wav_path in wav_paths:
        stem = os.path.splitext(os.path.basename(wav_path))[0]
        txt_path = os.path.join(data_root, stem + ".txt")
        if not os.path.exists(txt_path): continue
        try: pid = int(stem.split("_")[0])
        except (ValueError, IndexError): continue
        disease = diag_map.get(pid)
        if disease is None: continue

        is_unknown = 1 if disease in ICBHI_UNKNOWN_DISEASES else (0 if disease in ICBHI_KNOWN_DISEASES else -1)
        if is_unknown == -1: continue

        cycles = parse_annotation_file(txt_path)
        for c in cycles:
            row = {
                'wav_path': wav_path, 'stem': stem, 'patient_id': pid,
                'start': c['start'], 'end': c['end'], 'sound_label': c['label'],
                'disease_name': disease, 'is_unknown': is_unknown
            }
            if is_unknown == 0: known_rows.append(row)
            else: unknown_rows.append(row)

    df_known = pd.DataFrame(known_rows)
    df_unknown = pd.DataFrame(unknown_rows)
    return df_known, df_unknown

class RealICBHI_OWL_Dataset(Dataset):
    def __init__(self, df, cfg):
        self.df = df.reset_index(drop=True)
        self.cfg = cfg

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        spec = extract_log_mel(row['wav_path'], row['start'], row['end'], self.cfg)
        return (torch.from_numpy(spec),
                torch.tensor(row['sound_label'], dtype=torch.long),
                torch.tensor(row['is_unknown'], dtype=torch.long),
                row['patient_id'])

print("\n--- LOADING REAL ICBHI AUDIO FOR OWL STAGE 0 & STAGE 1 ---")
df_known, df_unknown = build_owl_dataset(CFG['data_root'], CFG)

known_pids = df_known['patient_id'].nunique()
unknown_pids = df_unknown['patient_id'].nunique()

print(f"Known Patients (Stage 0):   {len(df_known)} cycles across {known_pids} patients (COPD, Healthy, URTI)")
print(f"Unknown Patients (Stage 1): {len(df_unknown)} cycles across {unknown_pids} patients (Pneumonia, Bronchiectasis, etc.)")

known_dataset = RealICBHI_OWL_Dataset(df_known, CFG)
unknown_dataset = RealICBHI_OWL_Dataset(df_unknown, CFG)



--- LOADING REAL ICBHI AUDIO FOR OWL STAGE 0 & STAGE 1 ---
Loaded diagnosis map: /content/Respiratory_Sound_Database/Respiratory_Sound_Database/patient_diagnosis.csv (126 patients)
Known Patients (Stage 0):   6311 cycles across 104 patients (COPD, Healthy, URTI)
Unknown Patients (Stage 1): 587 cycles across 22 patients (Pneumonia, Bronchiectasis, etc.)


## Section 4: Load Architecture & Checkpoints (M2 & M13)


## Verify Checksums of Model Checkpoints

In [11]:
import hashlib

def calculate_sha256(filepath):
    """Calculates the SHA256 checksum of a file."""
    sha256_hash = hashlib.sha256()
    try:
        with open(filepath, "rb") as f:
            # Read and update hash string value in chunks
            for byte_block in iter(lambda: f.read(4096), b""):
                sha256_hash.update(byte_block)
        return sha256_hash.hexdigest()
    except FileNotFoundError:
        return "File not found"
    except Exception as e:
        return f"Error: {e}"


print("--- Verifying Checksums ---")

m2_path = '/content/M2_best_model.pth'
m13_path = '/content/M13_best_model.pth'

print(f"SHA256 for M2_best_model.pth: {calculate_sha256(m2_path)}")
print(f"SHA256 for M13_best_model.pth: {calculate_sha256(m13_path)}")

print("\n--- End Checksum Verification ---")

--- Verifying Checksums ---
SHA256 for M2_best_model.pth: d2406cee0ffab7ee12e5b31f72f9f7d3d63dc3475aea656770fd44cecabd061e
SHA256 for M13_best_model.pth: a1609c730f579455f8c3a68b1e3adc921f4773acc1b13c67d7cf40f24cb5a2e5

--- End Checksum Verification ---


In [12]:
# ============================================================
# Section 4: Load Architecture & Checkpoints (M2 & M13)
# ============================================================
import zipfile
import io

class ConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch, pool=(2, 2)):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=pool),
        )
    def forward(self, x): return self.block(x)

class M2_CNN(nn.Module):
    def __init__(self, num_classes=4, depth=5, base_width=48, dropout=0.4, fc_dim=128):
        super().__init__()
        channels = [base_width * (2 ** i) for i in range(depth)]
        blocks, in_ch = [], 1
        for out_ch in channels:
            blocks.append(ConvBlock(in_ch, out_ch))
            in_ch = out_ch
        self.encoder = nn.Sequential(*blocks)
        self.gap = nn.AdaptiveAvgPool2d((1, 1))
        self.dropout = nn.Dropout(dropout)
        self.head = nn.Sequential(
            nn.Linear(channels[-1], fc_dim),
            nn.ReLU(inplace=True),
            nn.Linear(fc_dim, num_classes),
        )
        self.embedding_dim = channels[-1]

    def forward(self, x):
        feat = self.gap(self.encoder(x)).flatten(1)
        return self.head(self.dropout(feat))

    def get_embedding(self, x):
        return self.gap(self.encoder(x)).flatten(1)

class PrototypicalDiseaseHead(nn.Module):
    def __init__(self, input_dim, embed_dim=256, num_classes=3):
        super().__init__()
        self.num_classes = num_classes
        self.projection = nn.Sequential(
            nn.Linear(input_dim, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(inplace=True),
            nn.Dropout(0.3),
            nn.Linear(512, embed_dim),
        )
        self.embed_dim = embed_dim

    def project(self, embeddings):
        z = self.projection(embeddings)
        return F.normalize(z, p=2, dim=-1)

    def compute_prototypes(self, support_embeddings, support_labels):
        prototypes = torch.zeros(self.num_classes, self.embed_dim, device=support_embeddings.device)
        for c in range(self.num_classes):
            mask = (support_labels == c)
            if mask.sum() > 0:
                prototypes[c] = support_embeddings[mask].mean(dim=0)
        return F.normalize(prototypes, p=2, dim=-1)

    def forward(self, query_embeddings, prototypes, temperature=0.1):
        dists = torch.cdist(query_embeddings, prototypes, p=2) ** 2
        return -dists / temperature

def smart_load_checkpoint(path, device):
    """
    Smart loader that handles:
    1. Standard PyTorch .pth checkpoints
    2. Zip bundles (e.g. M13_results_bundle.zip uploaded or renamed as .pth)
    """
    if not os.path.exists(path):
        raise FileNotFoundError(f"File not found: {path}")

    # 1. Check if the file is actually a zip bundle containing best_model.pth
    if zipfile.is_zipfile(path):
        try:
            with zipfile.ZipFile(path, 'r') as z:
                names = z.namelist()
                if 'best_model.pth' in names:
                    print(f"📦 Detected zip bundle at {path}. Extracting 'best_model.pth' from bundle...")
                    with z.open('best_model.pth') as f:
                        buffer = io.BytesIO(f.read())
                        return torch.load(buffer, map_location=device, weights_only=False)
                elif any(n.endswith('.pth') for n in names):
                    target_name = next(n for n in names if n.endswith('.pth'))
                    print(f"📦 Detected zip bundle at {path}. Extracting '{target_name}' from bundle...")
                    with z.open(target_name) as f:
                        buffer = io.BytesIO(f.read())
                        return torch.load(buffer, map_location=device, weights_only=False)
        except Exception as e:
            print(f"Zip bundle extraction note: {e}")

    # 2. Standard PyTorch load
    try:
        return torch.load(path, map_location=device, weights_only=False)
    except Exception:
        return torch.load(path, map_location=device, weights_only=True)

# Initialize Backbone & Disease Head
backbone = M2_CNN(num_classes=4, depth=CFG['m2_depth'], base_width=CFG['m2_base_width']).to(DEVICE)
proto_head = PrototypicalDiseaseHead(input_dim=768, embed_dim=CFG['proto_embed_dim'], num_classes=3).to(DEVICE)

# Load M2 Backbone Checkpoint
m2_loaded = False
m2_paths_to_try = [p for p in ['/content/M2_best_model.pth', '/content/M2_results_bundle.zip', CFG['m2_ckpt_path']] if p and os.path.exists(p)]
for p in m2_paths_to_try:
    try:
        fsize = os.path.getsize(p) / (1024 * 1024)
        ckpt = smart_load_checkpoint(p, DEVICE)
        state_dict = ckpt.get('model_state', ckpt)
        if isinstance(state_dict, dict) and 'model_state_dict' in state_dict:
            state_dict = state_dict['model_state_dict']
        backbone.load_state_dict(state_dict, strict=False)
        print(f"✅ Successfully loaded M2 Backbone from {p} ({fsize:.2f} MB)")
        m2_loaded = True
        break
    except Exception as e:
        fsize = os.path.getsize(p) / (1024 * 1024) if os.path.exists(p) else 0
        print(f"⚠️ Could not load M2 checkpoint at {p} ({fsize:.2f} MB): {e}")

if not m2_loaded:
    print(f"⚠️ Using default M2 Backbone weights.")

# Load M13 Disease Head Checkpoint
m13_loaded = False
m13_paths_to_try = [p for p in ['/content/M13_best_model.pth', '/content/M13_results_bundle.zip', CFG['m13_ckpt_path']] if p and os.path.exists(p)]
for p in m13_paths_to_try:
    try:
        fsize = os.path.getsize(p) / (1024 * 1024)
        m13_ckpt = smart_load_checkpoint(p, DEVICE)
        if isinstance(m13_ckpt, dict) and 'model_state' in m13_ckpt:
            proto_head.load_state_dict(m13_ckpt['model_state'], strict=False)
        elif isinstance(m13_ckpt, dict):
            proto_head.load_state_dict(m13_ckpt, strict=False)
        print(f"✅ Successfully loaded M13 Prototypical Head from {p} ({fsize:.2f} MB)")
        m13_loaded = True
        break
    except Exception as e:
        fsize = os.path.getsize(p) / (1024 * 1024) if os.path.exists(p) else 0
        print(f"⚠️ Could not load M13 checkpoint at {p} ({fsize:.2f} MB): {e}")

if not m13_loaded:
    print(f"⚠️ Using default M13 Prototypical Head weights.")

backbone.eval(); proto_head.eval()


✅ Successfully loaded M2 Backbone from /content/M2_best_model.pth (41.55 MB)
✅ Successfully loaded M13 Prototypical Head from /content/M13_best_model.pth (2.02 MB)


PrototypicalDiseaseHead(
  (projection): Sequential(
    (0): Linear(in_features=768, out_features=512, bias=True)
    (1): BatchNorm1d(512, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU(inplace=True)
    (3): Dropout(p=0.3, inplace=False)
    (4): Linear(in_features=512, out_features=256, bias=True)
  )
)

## Section 5: Cross-Task Disagreement Scoring


In [13]:
# ============================================================
# Section 5: Cross-Task Disagreement Scoring
# ============================================================

# Structural mapping matrix M_implied (Sound Event 4 -> Disease 3)
# [Normal, Crackle, Wheeze, Both] -> [COPD, Healthy, URTI]
M_IMPLIED = torch.tensor([
    [0.05, 0.90, 0.05],  # Normal -> High Healthy
    [0.60, 0.10, 0.30],  # Crackle -> High COPD / URTI
    [0.85, 0.05, 0.10],  # Wheeze -> High COPD
    [0.90, 0.02, 0.08],  # Both -> Very High COPD
], dtype=torch.float32, device=DEVICE)

def compute_cross_task_disagreement(model_backbone, head_proto, loader):
    """
    Computes cross-task structural disagreement score per patient.
    Disagreement S(x) = || P_sound_implied(x) - P_disease(x) ||_2
    """
    patient_disagreements = defaultdict(list)
    patient_energy_scores = defaultdict(list)
    patient_is_unknown = {}

    model_backbone.eval(); head_proto.eval()

    with torch.no_grad():
        for specs, sound_lbls, is_unk, pids in tqdm(loader, desc="Scoring consistency"):
            specs = specs.to(DEVICE)
            embeds = model_backbone.get_embedding(specs)

            # 1. Sound Event Head Logits & Softmax
            sound_logits = model_backbone(specs)
            p_sound = F.softmax(sound_logits, dim=-1)

            # 2. Implied Disease Profile
            p_sound_implied = torch.matmul(p_sound, M_IMPLIED)

            # 3. Direct Disease Head Predictions (Prototypical)
            z_proj = head_proto.project(embeds)
            prototypes = head_proto.compute_prototypes(z_proj, torch.randint(0, 3, (z_proj.size(0),), device=DEVICE))
            disease_logits = head_proto(z_proj, prototypes, CFG['proto_temperature'])
            p_disease = F.softmax(disease_logits, dim=-1)

            # 4. Disagreement Score: L2 Distance
            disagreement = torch.norm(p_sound_implied - p_disease, p=2, dim=-1).cpu().numpy()

            # 5. Energy OOD Baseline Score (M29 reference)
            T = 1.0
            energy = -T * torch.logsumexp(sound_logits / T, dim=-1).cpu().numpy()

            for i in range(len(pids)):
                pid = pids[i] if isinstance(pids[i], str) else int(pids[i])
                patient_disagreements[pid].append(disagreement[i])
                patient_energy_scores[pid].append(energy[i])
                patient_is_unknown[pid] = is_unk[i].item()

    patient_ids = sorted(list(patient_disagreements.keys()))
    y_true = np.array([patient_is_unknown[pid] for pid in patient_ids])
    s_disagree = np.array([np.mean(patient_disagreements[pid]) for pid in patient_ids])
    s_energy = np.array([np.mean(patient_energy_scores[pid]) for pid in patient_ids])

    return y_true, s_disagree, s_energy

print("\n--- COMPUTING CROSS-TASK DISAGREEMENT & ENERGY OOD SCORES ---")
y_known, s_disagree_known, s_energy_known = compute_cross_task_disagreement(backbone, proto_head, DataLoader(known_dataset, batch_size=CFG['batch_size']))
y_unknown, s_disagree_unk, s_energy_unk = compute_cross_task_disagreement(backbone, proto_head, DataLoader(unknown_dataset, batch_size=CFG['batch_size']))

y_all = np.concatenate([np.zeros(len(s_disagree_known)), np.ones(len(s_disagree_unk))])
scores_m15 = np.concatenate([s_disagree_known, s_disagree_unk])
scores_m29 = np.concatenate([s_energy_known, s_energy_unk])

print(f"\nTotal Evaluated Patients: {len(y_all)} ({len(s_disagree_known)} Known / {len(s_disagree_unk)} Unknown)")



--- COMPUTING CROSS-TASK DISAGREEMENT & ENERGY OOD SCORES ---


Scoring consistency: 100%|██████████| 19/19 [00:16<00:00,  1.17it/s]


Total Evaluated Patients: 126 (104 Known / 22 Unknown)


## Section 6: OOD Metrics & Baseline Comparison


In [14]:
# ============================================================
# Section 6: OOD Metrics & Baseline Comparison
# ============================================================

def compute_ood_metrics(y_true, scores):
    fpr, tpr, thresholds = roc_curve(y_true, scores)
    auroc = auc(fpr, tpr)
    precision, recall, _ = precision_recall_curve(y_true, scores)
    aupr = average_precision_score(y_true, scores)
    idx_tpr95 = np.argmin(np.abs(tpr - 0.95))
    fpr95 = float(fpr[idx_tpr95])
    return {'auroc': round(float(auroc), 4), 'aupr': round(float(aupr), 4), 'fpr95': round(fpr95, 4)}

m15_metrics = compute_ood_metrics(y_all, scores_m15)
m29_metrics = compute_ood_metrics(y_all, scores_m29)

print("\n" + "="*60)
print("OPEN-WORLD UNKNOWN DETECTION PERFORMANCE (PATIENT-LEVEL)")
print("="*60)
print(f"M15 Cross-Task Consistency Scorer (NOVELTY):")
print(f"  AUROC:  {m15_metrics['auroc']}")
print(f"  AUPR:   {m15_metrics['aupr']}")
print(f"  FPR95:  {m15_metrics['fpr95']}")
print(f"\nM29 Energy-based OOD Baseline:")
print(f"  AUROC:  {m29_metrics['auroc']}")
print(f"  AUPR:   {m29_metrics['aupr']}")
print(f"\nM6 OpenMax Reference Baseline:")
print(f"  AUROC:  0.4516")
print("="*60)



OPEN-WORLD UNKNOWN DETECTION PERFORMANCE (PATIENT-LEVEL)
M15 Cross-Task Consistency Scorer (NOVELTY):
  AUROC:  0.5782
  AUPR:   0.2202
  FPR95:  0.8942

M29 Energy-based OOD Baseline:
  AUROC:  0.6005
  AUPR:   0.2791

M6 OpenMax Reference Baseline:
  AUROC:  0.4516


## Section 7: Generate results_M15.json (§4 Schema)


In [15]:
# ============================================================
# Section 7: Generate results_M15.json (§4 Schema)
# ============================================================

results = {
    'meta': {
        'model_id': 'M15',
        'model_name': 'Cross-Task Consistency Scorer — v4',
        'member': 'B',
        'member_name': 'Member B (Disease Diagnosis & OWL)',
        'date_completed': datetime.datetime.now().strftime('%Y-%m-%d'),
        'is_augmented': False,
        'augmentation_method': 'none',
        'notes': (
            'v4: Cross-task consistency disagreement scorer evaluated on REAL ICBHI audio. '
            'Measures structural disagreement between Sound Event implied disease profile and Prototypical Disease Head. '
            'Patient-level evaluation across 104 Known Patients and 19 Unknown Patients. '
            'Replaces legacy un-bracketed M15_metrics.json schema with full §4 compliance.'
        ),
    },
    'config': CFG,
    'environment': {
        'platform': PLATFORM,
        'gpu_name': GPU_NAME,
        'pytorch_version': torch.__version__,
        'python_version': sys.version.split()[0],
    },
    'dataset_info': {
        'dataset': 'ICBHI_2017',
        'data_source': 'real_audio',
        'known_patients': known_pids,
        'unknown_patients': unknown_pids,
        'known_classes': CFG['disease_classes'],
        'unknown_classes': CFG['unknown_disease_classes'],
    },
    'best_metrics': m15_metrics,
    'baseline_comparisons': {
        'm29_energy_auroc': m29_metrics['auroc'],
        'm6_openmax_auroc': 0.4516,
    },
    'ablation': {
        'ablation_group': 'open_world_rejection_mechanism',
        'ablation_role': 'primary_novelty',
        'variable_changed': 'cross_task_disagreement_scoring',
    }
}

for out_dir in sorted(list({CFG['results_dir'], BASE_DIR})):
    os.makedirs(out_dir, exist_ok=True)
    rpath = os.path.join(out_dir, 'results_M15.json')
    with open(rpath, 'w') as f:
        json.dump(results, f, indent=2, default=str)
    print(f"✅ Saved results JSON: {rpath}")


✅ Saved results JSON: /content/results_M15.json
✅ Saved results JSON: /content/results_M15/results_M15.json


## Section 8: Visualizations & Plotting


In [16]:
# ============================================================
# Section 8: Visualizations & Plotting
# ============================================================

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ROC Curve
fpr_15, tpr_15, _ = roc_curve(y_all, scores_m15)
fpr_29, tpr_29, _ = roc_curve(y_all, scores_m29)

axes[0].plot(fpr_15, tpr_15, 'b-', label=f"M15 Consistency (AUROC = {m15_metrics['auroc']:.4f})", lw=2)
axes[0].plot(fpr_29, tpr_29, 'g--', label=f"M29 Energy (AUROC = {m29_metrics['auroc']:.4f})", lw=2)
axes[0].plot([0, 1], [0, 1], 'k:', label="Random (AUROC = 0.5000)")
axes[0].set_xlabel('False Positive Rate (Knowns rejected)')
axes[0].set_ylabel('True Positive Rate (Unknowns detected)')
axes[0].set_title('ROC Curve — Unknown Disease Detection')
axes[0].legend(loc='lower right')
axes[0].grid(True, alpha=0.3)

# Distribution Plot
sns.kdeplot(s_disagree_known, label="Known Patients (Stage 0)", ax=axes[1], fill=True, color='blue', alpha=0.3)
sns.kdeplot(s_disagree_unk, label="Unknown Patients (Stage 1)", ax=axes[1], fill=True, color='red', alpha=0.3)
axes[1].set_xlabel('Cross-Task Disagreement Score S(x)')
axes[1].set_ylabel('Density')
axes[1].set_title('Disagreement Score Distribution')
axes[1].legend()

plt.tight_layout()
for out_dir in sorted(list({CFG['results_dir'], BASE_DIR})):
    fig.savefig(os.path.join(out_dir, 'consistency_scorer_results.png'), dpi=150, bbox_inches='tight')
print("Saved plots: consistency_scorer_results.png")
plt.close()


Saved plots: consistency_scorer_results.png


## Section 9: Bundle & Download Output Files (Kaggle & Colab)


In [17]:
# ============================================================
# Section 9: Bundle & Download Output Files (Kaggle & Colab)
# ============================================================
import shutil
import base64
from IPython.display import display, HTML

zip_file_name = "M15_results_bundle"
zip_target_path = os.path.join(BASE_DIR, zip_file_name)
if os.path.exists(zip_target_path + ".zip"): os.remove(zip_target_path + ".zip")

archive_file = shutil.make_archive(zip_target_path, 'zip', CFG['results_dir'])
archive_size_mb = os.path.getsize(archive_file) / (1024 * 1024)

print("\n" + "=" * 60)
print("M15 RESULTS DOWNLOAD BUNDLE CREATED")
print("=" * 60)
print(f"Zip bundle path: {archive_file} ({archive_size_mb:.2f} MB)")

try:
    with open(archive_file, 'rb') as f:
        b64_data = base64.b64encode(f.read()).decode('utf-8')
    b64_href = f"data:application/zip;base64,{b64_data}"
    html_button = f'''
<div style="background-color: #e7f5ff; border: 1px solid #74c0fc; padding: 16px; border-radius: 8px; margin: 12px 0;">
  <h3 style="margin-top:0; color: #1864ab;">📥 M15 Results Bundle Ready ({archive_size_mb:.2f} MB)</h3>
  <p><a href="{b64_href}" download="M15_results_bundle.zip" style="display: inline-block; background-color: #1c7ed6; color: white; padding: 12px 24px; text-decoration: none; border-radius: 6px; font-weight: bold; font-size: 15px;">
    ⬇️ Click Here to Download M15_results_bundle.zip
  </a></p>
</div>
'''
    display(HTML(html_button))
except Exception as e:
    print(f"Base64 download note: {e}")



M15 RESULTS DOWNLOAD BUNDLE CREATED
Zip bundle path: /content/M15_results_bundle.zip (0.13 MB)
